In [1]:
%load_ext autoreload
%autoreload 2

In [18]:
import numpy as np
import torch
import torch.nn as nn

from nichejepa.models.gene_transformer import *

from nichejepa.utils.tensors import (repeat_interleave_batch,
                                     trunc_normal_,)

In [3]:
pos_emb = get_1d_sincos_pos_embed_from_grid(embed_dim=4,
                                            pos=np.arange(5, dtype=float))
pos_emb

array([[ 0.        ,  0.        ,  1.        ,  1.        ],
       [ 0.84147098,  0.00999983,  0.54030231,  0.99995   ],
       [ 0.90929743,  0.01999867, -0.41614684,  0.99980001],
       [ 0.14112001,  0.0299955 , -0.9899925 ,  0.99955003],
       [-0.7568025 ,  0.03998933, -0.65364362,  0.99920011]])

In [4]:
pos_emb = get_1d_sincos_pos_embed(embed_dim=4,
                                  grid_size=5,
                                  cls_token=True)
pos_emb

array([[ 0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  1.        ,  1.        ],
       [ 0.84147098,  0.00999983,  0.54030231,  0.99995   ],
       [ 0.90929743,  0.01999867, -0.41614684,  0.99980001],
       [ 0.14112001,  0.0299955 , -0.9899925 ,  0.99955003],
       [-0.7568025 ,  0.03998933, -0.65364362,  0.99920011]])

In [20]:
drop_prob = 0.1
training = True

x = torch.tensor([[12, 5, 4, 28, 64, 32, 16, 8, 55, 3],
                  [4, 3, 12, 16, 22, 5, 44, 33, 14, 24]])

drop_path(x,
          drop_prob=drop_prob,
          training=training)

RuntimeError: "check_uniform_bounds" not implemented for 'Long'

### GeneTransformerEncoder Forward Pass

In [5]:
vocab_size = 100
embed_dim = 4
seq_len = 10
depth = 12
pos_learnable = False
has_cls = True
drop_path_rate = 0.0
init_std = 0.02

In [6]:
# Initialize gene embeddings
gene_embed = nn.Embedding(
    vocab_size + (1 if has_cls else 0), # vocab_size incl. <pad>
    embed_dim,
    padding_idx=0)
gene_embed(torch.tensor([99]))

tensor([[ 0.3886,  0.3611, -0.8015,  0.9050]], grad_fn=<EmbeddingBackward0>)

In [7]:
# Initialize segment embeddings
seg_embed = nn.Embedding(2 + 1, # incl. <pad>
                         embed_dim,
                         padding_idx=0)
seg_embed(torch.tensor([0, 1, 2]))

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0325,  0.1703, -1.0835,  1.1582],
        [-0.0531, -0.2647, -0.7567,  1.0447]], grad_fn=<EmbeddingBackward0>)

In [8]:
# Retrieve positional embeddings
if pos_learnable:
    pos_embed = nn.Parameter(
        torch.zeros(1, seq_len + (1 if has_cls else 0), embed_dim),
        requires_grad=True)
    trunc_normal_(pos_embed, std=init_std)
    if has_cls:
        pos_embed.data[0, 0, :] = 0
else:
    pos_embed = nn.Parameter(
        torch.zeros(1, seq_len + (1 if has_cls else 0), embed_dim),
        requires_grad=False)
    pos_embed_temp = get_1d_sincos_pos_embed(pos_embed.shape[-1],
                                             int(seq_len),
                                             cls_token=has_cls)
    pos_embed.data.copy_(
        torch.from_numpy(pos_embed_temp).float().unsqueeze(0))
pos_embed

Parameter containing:
tensor([[[ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  1.0000,  1.0000],
         [ 0.8415,  0.0100,  0.5403,  0.9999],
         [ 0.9093,  0.0200, -0.4161,  0.9998],
         [ 0.1411,  0.0300, -0.9900,  0.9996],
         [-0.7568,  0.0400, -0.6536,  0.9992],
         [-0.9589,  0.0500,  0.2837,  0.9988],
         [-0.2794,  0.0600,  0.9602,  0.9982],
         [ 0.6570,  0.0699,  0.7539,  0.9976],
         [ 0.9894,  0.0799, -0.1455,  0.9968],
         [ 0.4121,  0.0899, -0.9111,  0.9960]]])

In [9]:
dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
dpr

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

### GeneTransformerEncoder Embedding Retrieval

In [10]:
vocab_size = 100
embed_dim = 12
seq_len = 10
depth = 12
pos_learnable = True
has_cls = True

In [11]:
encoder = GeneTransformerEncoder(vocab_size=vocab_size,
                                 embed_dim=embed_dim,
                                 seq_len=seq_len,
                                 pos_learnable=pos_learnable,
                                 has_cls=has_cls)

In [12]:
if has_cls:
    x = torch.tensor([[100, 12, 5, 4, 28, 64, 32, 16, 8, 55, 3],
                      [100, 4, 3, 12, 16, 22, 5, 44, 33, 14, 24]])
    seg_label = torch.tensor([[0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2],
                              [0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2]])
else:
    x = torch.tensor([[12, 5, 4, 28, 64, 32, 16, 8, 55, 3],
                      [4, 3, 12, 16, 22, 5, 44, 33, 14, 24]])
    seg_label = torch.tensor([[1, 1, 1, 1, 1, 2, 2, 2, 2, 2],
                              [1, 1, 1, 1, 1, 2, 2, 2, 2, 2]])

In [13]:
pos_embeds = encoder.return_pos_emb(x)
print(pos_embeds[0].shape)
print(pos_embeds[0])

torch.Size([2, 11, 12])
tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00],
         [ 1.2355e-02, -2.3010e-02,  8.2211e-03,  2.5494e-02, -1.6048e-03,
           2.8943e-03, -2.4445e-02, -2.7349e-02, -5.4024e-02, -1.8526e-02,
           1.4072e-02, -2.1085e-02],
         [-1.5393e-02, -2.3486e-03,  3.1873e-02,  2.8099e-02,  1.9450e-02,
          -2.3372e-02,  1.8914e-02,  3.2293e-02, -2.9696e-02, -7.7141e-03,
          -1.3853e-03, -3.3021e-03],
         [-1.9069e-02,  1.5562e-03, -1.8923e-02, -3.0239e-02,  4.8298e-03,
          -1.3761e-02,  1.5019e-02,  3.8481e-02,  1.1468e-02, -2.7159e-02,
          -2.3069e-02,  1.8664e-02],
         [ 9.9355e-03,  3.4599e-03, -8.4652e-03, -2.3457e-03,  2.1908e-02,
          -2.2861e-03, -3.2913e-02, -1.2396e-02,  4.6330e-02, -2.6818e-03,
          -1.3892e-02,  1.5164e-02],
         [ 5.8597e-05, -3.4147e-03, -1.61

In [14]:
gene_embeds = encoder.return_gene_emb(x)
print(gene_embeds[0].shape)
print(gene_embeds[0])

torch.Size([2, 11, 12])
tensor([[[-0.1524,  0.7110, -0.9137,  1.0956, -1.6392,  0.2688,  0.3496,
          -0.8542, -0.4997,  0.5988, -1.5334, -0.4865],
         [-0.1924, -1.2466,  0.2317, -0.8607,  0.5865,  2.7729,  0.8435,
          -0.1977,  0.0756,  1.1569, -0.5088,  2.3436],
         [-1.2523, -0.8022,  0.4693,  0.6585,  0.6316,  0.4823,  0.6944,
           0.3934,  0.2693,  0.5077,  0.9964, -1.1958],
         [-0.1465,  1.5002,  0.5096, -1.8781,  1.0948,  1.0666,  1.7202,
          -1.0140, -1.0083,  0.5655,  0.8062,  1.9770],
         [ 0.2847,  0.4613,  0.5289, -0.8173, -0.7060,  1.4964, -0.5553,
          -0.4731, -0.0368, -0.3605, -0.2566, -0.7452],
         [ 1.2759,  0.4012,  0.0142,  2.1004, -1.1953, -0.2197, -0.7751,
          -0.2520, -1.4847,  1.5515,  0.5760, -1.7360],
         [-1.0820, -0.4297, -0.5467, -0.2102, -0.8424, -0.0685, -1.1631,
           0.0657, -0.4367,  1.8713,  0.2928,  0.3553],
         [ 0.2568, -0.1285,  0.5269,  0.0937, -1.4065, -0.0051,  0.2783,


In [15]:
embeds = encoder.return_multi_layer_emb(x,
                                        seg_label)
print(len(embeds))
print(embeds[-1].shape)
print(embeds[-1])

12
torch.Size([2, 11, 12])
tensor([[[ 0.1147,  1.1378, -0.7848,  1.6008, -1.6325,  0.6177,  0.7167,
          -0.7142, -0.2896,  1.0099, -1.5053, -0.2711],
         [-0.2520, -1.5147, -0.5604, -0.8871, -0.7960,  2.1171,  0.6661,
           0.4289, -0.4740,  0.6811, -0.6880,  1.2790],
         [-1.2183, -1.2792, -0.2066,  0.6693, -0.7458,  0.6562,  1.0459,
           1.5176, -0.1290,  0.4998,  0.9008, -1.7108],
         [-0.2946,  0.7339, -0.4431, -1.9975, -0.4750,  0.8604,  1.5771,
          -0.2041, -1.4792,  0.2222,  0.3430,  1.1569],
         [ 0.7489,  0.3346,  0.0762, -0.6917, -2.0306,  2.1819, -0.0602,
           0.9070, -0.1171, -0.1464, -0.1686, -1.0338],
         [ 1.0629,  0.0787, -0.3921,  1.5382, -1.6923,  0.0582, -0.2428,
           0.6358, -1.1899,  1.1775,  0.3985, -1.4327],
         [-0.8135, -0.1261, -1.2189,  0.3216,  0.3632,  1.2646, -1.2083,
          -0.7931, -1.1357,  1.8760,  0.6280,  0.8421],
         [ 0.5608,  0.3002, -0.4292,  1.0725, -0.0848,  2.2471,  0.068